# Advanced object-oriented programming 🧠

## What you will learn in this course 🧐🧐

You understand classes, objects, attributes, and methods from the fundamentals lecture. These concepts work well for simple systems but operational complexity demands more sophisticated patterns. Real systems require hierarchies of related types, uniform interfaces for different implementations, and flexible composition of components.

By the end of this course, you will:

- Build inheritance hierarchies for specialized types
- Implement polymorphism for uniform interfaces
- Use composition to build complex systems from simple parts
- Apply abstraction to hide implementation complexity
- Integrate objects with Python's protocols through special methods

In [1]:
import random
from typing import List, Dict, Optional
from abc import ABC, abstractmethod

## Building fleet command systems

Your basic OOP skills created working pilot management systems. Now the Rebel Alliance faces new challenges. The fleet contains different ship types that share common capabilities like acceleration and navigation but implement specialized systems differently. 

Squadrons must coordinate pilots and ships through uniform interfaces.

- The procedural approach would duplicate acceleration code across every ship type. 
- Basic OOP would create independent classes, still duplicating shared behavior. 
- Advanced OOP patterns solve these problems through inheritance, polymorphism, and composition.

## Abstraction and inheritance

<img src = "https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Abstraction.png"/>

### Abstraction

**Abstraction = focus on *what* an object can do, not *how* it does it.**

You define an interface like:

> A `Ship` can `launch()`, `land()`, `report_status()`

Without caring (for now) whether it’s an X-Wing, Y-Wing, or Falcon.
From the outside, you just use these methods. The internal details are hidden.

### Inheritance

**Inheritance = reuse + specialization.**

<img src = ""/>

* You create a **base class** (e.g. `Ship`) with common attributes and methods.
* Then you create **child classes** (e.g. `XWing`, `Falcon`) that:

  * inherit everything from `Ship`,
  * add or override behavior (e.g. `hyperjump()` only for the Falcon).

So you avoid rewriting the same code for every type of ship.

### Abstract base classes in Python

Python lets you formalize this idea with:

* `ABC` (Abstract Base Class)
* `@abstractmethod`

You define a class that says:

> Any subclass of me **must** implement these methods.

For example, an abstract `Starship` might require every child to implement `accelerate()` and `take_damage()`.

It’s like a contract: 

> If you inherit from me, you promise to provide this behavior.

In [2]:
class Starship(ABC):
    """Abstract base class defining starship interface"""
    
    def __init__(self, designation, max_speed):
        """Initialize common starship attributes"""
        self.designation = designation
        self.max_speed = max_speed
        self.current_speed = 0
        self.shields = 100
        self.hull_integrity = 100
    
    def accelerate(self, amount):
        """Concrete method - shared implementation"""
        self.current_speed = min(self.current_speed + amount, self.max_speed)
        return f"{self.designation}: Speed {self.current_speed}/{self.max_speed}"
    
    def take_damage(self, amount):
        """Concrete method - common damage processing"""
        if self.shields > 0:
            shield_damage = min(amount, self.shields)
            self.shields -= shield_damage
            amount -= shield_damage
        
        if amount > 0:
            self.hull_integrity -= amount
        
        return self.is_destroyed()
    
    def is_operational(self):
        """Check if ship can operate"""
        return self.hull_integrity > 20
    
    def is_destroyed(self):
        """Check if ship destroyed"""
        return self.hull_integrity <= 0
    
    @abstractmethod
    def engage_combat(self):
        """Abstract method - subclasses must implement"""
        pass
    
    @abstractmethod
    def get_capabilities(self):
        """Abstract method - type-specific info"""
        pass

The Fighter class extends Starship by inheriting common functionality and adding combat-specific capabilities.

In [3]:
class Fighter(Starship):
    """Combat ship extending Starship with weapons"""
    
    def __init__(self, designation, max_speed=1050):
        # Call parent constructor
        super().__init__(designation, max_speed)
        
        # Add fighter-specific attributes
        self.ammunition = 100
        self.torpedo_count = 6
        self.kills = 0
    
    def fire_weapons(self):
        """Fighter-specific combat capability"""
        if not self.is_operational():
            return {'success': False, 'message': 'Ship not operational'}
        
        if self.ammunition <= 0:
            return {'success': False, 'message': 'Out of ammunition'}
        
        self.ammunition -= 10
        hit = random.random() > 0.3
        
        if hit:
            self.kills += 1
            return {
                'success': True,
                'message': f"{self.designation}: Target destroyed!",
                'kills': self.kills
            }
        
        return {'success': False, 'message': 'Missed target'}
    
    def launch_torpedo(self):
        """Fire proton torpedo"""
        if self.torpedo_count <= 0:
            return {'success': False, 'message': 'No torpedoes'}
        
        self.torpedo_count -= 1
        hit = random.random() > 0.2
        return {'success': hit, 'torpedoes': self.torpedo_count}
    
    def engage_combat(self):
        """Implement abstract method - fighter combat"""
        return f"{self.designation} engaging with laser cannons and torpedoes"
    
    def get_capabilities(self):
        """Implement abstract method - fighter info"""
        return {
            'role': 'Space Superiority',
            'ammunition': self.ammunition,
            'torpedoes': self.torpedo_count,
            'kills': self.kills
        }
    
    def __str__(self):
        return f"{self.designation} (Fighter) - Hull {self.hull_integrity}%, Shields {self.shields}%"

In [4]:
# Create fighter inheriting from Starship
red_five = Fighter("Red Five", max_speed=1050)

# Use inherited methods
print(red_five.accelerate(300))
print(red_five.accelerate(200))

Red Five: Speed 300/1050
Red Five: Speed 500/1050


In [5]:
# Use fighter-specific methods
print("\n" + str(red_five.fire_weapons()))
print(str(red_five.launch_torpedo()))


{'success': True, 'message': 'Red Five: Target destroyed!', 'kills': 1}
{'success': True, 'torpedoes': 5}


In [6]:
# Test damage system
print("\nTaking damage...")
destroyed = red_five.take_damage(45)
print(f"Destroyed: {destroyed}")
print(red_five)


Taking damage...
Destroyed: False
Red Five (Fighter) - Hull 100%, Shields 55%


In [7]:
# Abstract methods implemented
print("\n" + red_five.engage_combat())
print(f"Capabilities: {red_five.get_capabilities()}")


Red Five engaging with laser cannons and torpedoes
Capabilities: {'role': 'Space Superiority', 'ammunition': 90, 'torpedoes': 5, 'kills': 1}


**Inheritance creates a type hierarchy.**

A Fighter is a Starship. The Fighter class inherits `accelerate()`, `take_damage()`, and status checking methods without duplication. It adds `fire_weapons()` and `launch_torpedo()` for combat. It implements abstract methods `engage_combat()` and `get_capabilities()` with fighter-specific behavior.

<Note type="important">

The `super()` function accesses parent methods. Calling `super().__init__(designation, max_speed)` invokes the parent constructor before adding child-specific initialization.

</Note>

In [20]:
# Exemple simple : HÉRITAGE vs COMPOSITION

# ===== HÉRITAGE =====
# Une classe "enfant" hérite des propriétés et méthodes d'une classe "parent"

class Animal:
    """Classe parent - contient ce que tous les animaux ont en commun"""
    def __init__(self, name):
        self.name = name
    
    def faire_bruit(self):
        return "Son générique"

class Chien(Animal):
    """Classe enfant - hérite de Animal"""
    def faire_bruit(self):
        return f"{self.name} dit: Ouaf ! 🐶"

class Chat(Animal):
    """Classe enfant - hérite de Animal"""
    def faire_bruit(self):
        return f"{self.name} dit: Miaou ! 🐱"

# Utilisation de l'héritage
rex = Chien("Rex")
minou = Chat("Minou")

print("=== HÉRITAGE ===")
print(rex.faire_bruit())      # Rex dit: Ouaf ! 🐶
print(minou.faire_bruit())    # Minou dit: Miaou ! 🐱


# ===== COMPOSITION =====
# Un objet est "composé" d'autres objets plutôt que d'en hériter

class Moteur:
    """Composant indépendant"""
    def demarrer(self):
        return "Vroom ! 🔧"

class Voiture:
    """Une voiture est composée d'un moteur (elle contient un moteur)"""
    def __init__(self, brand, moteur):
        self.brand = brand
        self.moteur = moteur  # La voiture HAS-A un moteur
    
    def demarrer(self):
        return f"{self.brand} démarre: {self.moteur.demarrer()}"

# Utilisation de la composition
moteur_diesel = Moteur()
ma_voiture = Voiture("Peugeot", moteur_diesel)

print("\n=== COMPOSITION ===")
print(ma_voiture.demarrer())  # Peugeot démarre: Vroom ! 🔧


# ===== RÉSUMÉ =====
# HÉRITAGE: Chien IS-A Animal (un chien est un animal)
# COMPOSITION: Voiture HAS-A Moteur (une voiture a un moteur)

=== HÉRITAGE ===
Rex dit: Ouaf ! 🐶
Minou dit: Miaou ! 🐱

=== COMPOSITION ===
Peugeot démarre: Vroom ! 🔧


## Polymorphism

Polymorphism enables writing code once that works with multiple types. 

<img src = "https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Polymorphisme.png"/>

> same code, different behaviors depending on the object

You write code that works with **any object** that has the right methods, without caring about its exact class.

Example:

In [8]:
def launch_ship(ship):
    """Duck-typed polymorphism: anything with .launch() works"""
    ship.launch()


In [9]:
class XWing:
    def launch(self):
        print("X-Wing launching in attack formation!")

class CargoShip:
    def launch(self):
        print("Cargo ship launching with heavy load.")


fleet = [XWing(), CargoShip()]

for ship in fleet:
    launch_ship(ship)   


X-Wing launching in attack formation!
Cargo ship launching with heavy load.




Same function `launch_ship(ship)`:

* If `ship` is an `XWing` → it prints attack formation.
* If `ship` is a `CargoShip` → it prints heavy load.

You didn’t write `if type is XWing then ...`, `if type is CargoShip then ...`
You just said: **“I need something that has a `launch()` method.”**
Each object does its own version of `launch()`.

In Python, that’s often called **duck typing**:

> If it walks like a duck and quacks like a duck, I treat it like a duck. 🦆


In [10]:
class Transport(Starship):
    """Non-combat cargo vessel
    Transport inherits from Starship. 
    So it’s a kind of Starship, but specialized for cargo, not combat.
    It calls super().__init__ to reuse the initialization from Starship.
    Then it adds its own specific stuff:
    - cargo_capacity
    - cargo_loaded
    So: same base behavior, plus transport-specific attributes.
    """
    
    def __init__(self, designation, max_speed=650, cargo_capacity=500):
        super().__init__(designation, max_speed)
        self.cargo_capacity = cargo_capacity
        self.cargo_loaded = 0
    
    def load_cargo(self, amount):
        """Load cargo within capacity"""
        available = self.cargo_capacity - self.cargo_loaded
        if amount > available:
            return {'success': False, 'message': f'Only {available} tons available'}
        
        self.cargo_loaded += amount
        return {'success': True, 'loaded': self.cargo_loaded}
    
    def engage_combat(self):
        """Implement abstract method - transport has no weapons"""
        return f"{self.designation} executing evasive maneuvers - no weapons"
    
    def get_capabilities(self):
        """Implement abstract method - transport info"""
        return {
            'role': 'Logistics',
            'cargo': f"{self.cargo_loaded}/{self.cargo_capacity}"
        }
    
    def __str__(self):
        return f"{self.designation} (Transport) - Cargo {self.cargo_loaded}/{self.cargo_capacity} tons"

In [11]:
def fleet_combat(ships: List[Starship]):
    """
    Coordinate combat across different ship types
    
    Accepts any Starship objects. Each responds polymorphically.
    """
    print("=== FLEET COMBAT SEQUENCE ===")
    for ship in ships:
        # Polymorphic call - behavior varies by ship type
        print(ship.engage_combat())
    print("="*40)


Here:

* `fleet_combat` doesn’t know *which* kind of ship it receives.
* It simply calls `ship.engage_combat()`.
* Each object runs **its own implementation** of `engage_combat`:

  * Fighters fire weapons,
  * Transports perform evasive maneuvers,
  * Bombers might launch heavy payloads, etc.

> Same function call, different behavior depending on the actual object type.
> That’s polymorphism.


In [12]:
def fleet_status(ships: List[Starship]):
    """Report status across fleet"""
    print("\n=== FLEET STATUS ===")
    for ship in ships:
        print(f"{ship}")
        caps = ship.get_capabilities()
        print(f"  Capabilities: {caps}")
    print("="*40)

This function shows **two more polymorphic calls**:

* `print(f"{ship}")` → calls `ship.__str__()`, and each ship class can define its own string representation.
* `ship.get_capabilities()` → each ship returns different information (weapons, cargo, role, etc.) using the same method name.

> The fleet code only assumes that ships implement `__str__` and `get_capabilities`.
> It doesn’t need `if` statements or type checks. New ship types can be added later, and `fleet_status` will work with them unchanged.

That’s the power of polymorphism:
**you write generic code once, and new classes “plug in” by respecting the same interface.**


In [13]:
# Create mixed fleet
fleet = [
    Fighter("Red Five"),
    Fighter("Red Two"),
    Transport("Cargo One", cargo_capacity=600)
]

In [14]:
# Polymorphic operations work across all types
fleet_combat(fleet)
fleet_status(fleet)

=== FLEET COMBAT SEQUENCE ===
Red Five engaging with laser cannons and torpedoes
Red Two engaging with laser cannons and torpedoes
Cargo One executing evasive maneuvers - no weapons

=== FLEET STATUS ===
Red Five (Fighter) - Hull 100%, Shields 100%
  Capabilities: {'role': 'Space Superiority', 'ammunition': 100, 'torpedoes': 6, 'kills': 0}
Red Two (Fighter) - Hull 100%, Shields 100%
  Capabilities: {'role': 'Space Superiority', 'ammunition': 100, 'torpedoes': 6, 'kills': 0}
Cargo One (Transport) - Cargo 0/600 tons
  Capabilities: {'role': 'Logistics', 'cargo': '0/600'}


In [15]:
# Each ship type has specialized capabilities
print("\n=== SPECIALIZED OPERATIONS ===")
fleet[0].fire_weapons()  # Fighter method
print(fleet[2].load_cargo(200))  # Transport method


=== SPECIALIZED OPERATIONS ===
{'success': True, 'loaded': 200}


<Note type ="tip">

Polymorphism delivers extensibility. The `fleet_combat()` and `fleet_status()` functions work with any Starship subclass. Add new ship types without modifying these functions. They depend only on the Starship interface. Each ship implements `engage_combat()` and `get_capabilities()` differently, but external code uses the same interface.

</Note>

In [21]:
# Exemple simple du polymorphisme

# Le polymorphisme : la même méthode, des comportements différents

class Chat:
    def faire_bruit(self):
        return "Miaou ! 🐱"

class Chien:
    def faire_bruit(self):
        return "Ouaf ! 🐶"

class Canard:
    def faire_bruit(self):
        return "Coin coin ! 🦆"

# Polymorphisme : même appel, résultats différents
animaux = [Chat(), Chien(), Canard()]

print("=== Sans polymorphisme (répétitif) ===")
chat = Chat()
print(chat.faire_bruit())

chien = Chien()
print(chien.faire_bruit())

canard = Canard()
print(canard.faire_bruit())

print("\n=== Avec polymorphisme (magique) ===")
for animal in animaux:
    print(animal.faire_bruit())  # Même appel, comportements différents !

=== Sans polymorphisme (répétitif) ===
Miaou ! 🐱
Ouaf ! 🐶
Coin coin ! 🦆

=== Avec polymorphisme (magique) ===
Miaou ! 🐱
Ouaf ! 🐶
Coin coin ! 🦆


## Composition

Composition is a design principle where a class is composed of one or more objects from other classes, allowing for a flexible and modular structure.

<img src = "https://data-analytics-fullstack-assets.s3.eu-west-3.amazonaws.com/M05-Python_Programming/M2_D3_Composition.png"/>

When we say:

> A Squadron **has** Pilots and **has** Fighters

we’re saying:

* A **Squadron** is *not* a kind of Pilot.
* A **Squadron** is *not* a kind of Fighter.
* Instead, a **Squadron** is an object that **contains** pilots and fighters.

That’s composition:

> building bigger objects by assembling smaller ones inside them.

In code:

In [16]:
class Pilot:
    def __init__(self, name, experience):
        self.name = name
        self.experience = experience


class Fighter:
    def __init__(self, model, callsign):
        self.model = model      # e.g. "X-Wing"
        self.callsign = callsign  # e.g. "Red Five"


class Squadron:
    def __init__(self, name):
        self.name = name
        self.pilots = []     # Squadron HAS pilots
        self.fighters = []   # Squadron HAS fighters

    def add_pilot(self, pilot):
        self.pilots.append(pilot)

    def add_fighter(self, fighter):
        self.fighters.append(fighter)

Here:

* `Squadron` **has** a list of `Pilot` objects
* `Squadron` **has** a list of `Fighter` objects

That’s composition.

Composition becomes powerful when you combine it with **Polymorphism**. Because `Squadron` just **holds** pilots and fighters, it doesn’t care *what exact classes* they come from, as long as they behave correctly.

You can do things like:

In [17]:
class XWing(Fighter):
    pass

class AWing(Fighter):
    pass


In [18]:
red_squadron = Squadron("Red Squadron")
red_squadron.add_fighter(XWing("T-65B", "Red Five"))
red_squadron.add_fighter(AWing("RZ-1", "Red Two"))
red_squadron.add_pilot(Pilot("Luke Skywalker", experience=5))
red_squadron.add_pilot(Pilot("Wedge Antilles", experience=4))

In [19]:
print(f"Squadron: {red_squadron.name}")
for pilot in red_squadron.pilots:
    print(f" Pilot: {pilot.name}, Experience: {pilot.experience} years")
for fighter in red_squadron.fighters:
    print(f" Fighter: {fighter.model}, Callsign: {fighter.callsign}")

Squadron: Red Squadron
 Pilot: Luke Skywalker, Experience: 5 years
 Pilot: Wedge Antilles, Experience: 4 years
 Fighter: T-65B, Callsign: Red Five
 Fighter: RZ-1, Callsign: Red Two


`Squadron` works with:

* `XWing`
* `AWing`
* `YWing`
* future experimental prototypes you haven’t invented yet

…because it’s composed of generic `Fighter` objects (or anything that looks/acts like one), not tied to a specific subclass.

<Note type="tip">

* **Composition** = “has-a”

  * A Squadron **has** pilots
  * A Squadron **has** fighters
  
* **Inheritance** = “is-a”

  * An XWing **is a** Fighter
  * A MedicalDroid **is a** Droid

Composition is great when:
* You’re assembling systems from interchangeable parts
* You want to be able to **swap, add, or remove** pieces at runtime
* You want each part to evolve independently
  
</Note>


## Resources 📚📚

- [Python Inheritance and Composition](https://realpython.com/inheritance-composition-python/)
- [Abstract Base Classes](https://docs.python.org/3/library/abc.html)
- [Special Method Names](https://docs.python.org/3/reference/datamodel.html#special-method-names)
- [Composition vs Inheritance](https://realpython.com/inheritance-composition-python/#whats-composition)
- [SOLID Principles](https://en.wikipedia.org/wiki/SOLID)